# Enigma Machine Simulator
## Introduction
This notebook walks through an object-oriented simulation of the Enigma machine written in Python, and then uses it to break a set of Enigma-encrypted messages with a crib-based brute-force search.

The code lives in two modules next to this notebook:
* `enigma.py`: the simulator (`PlugLead`, `Plugboard`, `Rotor`, `EnigmaMachine`).
* `code_breaking.py`: the brute-force search routines used in Part Two.

## Part One – Simulation
The simulator is object-oriented: each physical component of the machine (plug lead, plugboard, rotor, reflector, the machine itself) is modelled by a class. Because the code is in a separate module, the cell below simply imports it.


In [1]:
from enigma import *


### Plug Leads
The simulation starts with the plugboard, specifically the leads. Each lead in the enigma machine connects two plugs in the plugboard. If we think of the functionality of the plugboard itself in terms of how it encodes a single character, the plugboard *aggregates* the leads, so it makes sense to have a class representing the lead objects.

The `PlugLead` class models a single lead. Its constructor should take a string of length two, which represents the two characters this lead should connect. So the following code:
```python3
lead = PlugLead("AG")
```
creates a lead which connects A to G. Leads are reversible, so this lead also connects G to A.

A lead exposes a method `encode(c)`, which takes a single character `c` and returns the result of passing it through the lead. 

So, with the `lead` object we created above, `lead.encode("G")` should return `"A"`.

`lead.encode("D")` should return `"D"` – this lead had no impact on the letter D, it only connects A and G. 

A letter cannot be connected to itself, since there is only one plug for each letter on the plugboard. The simulator respects the physical limitations of the Enigma machine, so constructing an invalid lead raises an error (a `ValueError`).

***Note:*** at this point we'll note that, of course, whenever the Enigma machine *encodes*, it also *decodes* for the exact same settings. The method name `encode(…)` is used throughout, and the same method is used for encoding and decoding.

In [2]:
lead = PlugLead("AG")
assert(lead.encode("A") == "G")
assert(lead.encode("D") == "D")

lead = PlugLead("DA")
assert(lead.encode("A") == "D")
assert(lead.encode("D") == "A")

### Plugboard
The plugboard houses the leads. Only its *interface* matters to the rest of the machine: leads are added through the method `.add(…)` as shown below, and `.encode(…)` returns the result of passing a character through the entire plugboard.

The real Enigma machine only came with 10 leads to connect plugs, so the plugboard accepts at most 10 leads and handles the obvious physical incompatibilities (e.g. a plug already in use) with warnings and errors.


In [3]:
plugboard = Plugboard()

plugboard.add(PlugLead("SZ"))
plugboard.add(PlugLead("GT"))
plugboard.add(PlugLead("DV"))
plugboard.add(PlugLead("KU"))

assert(plugboard.encode("K") == "U")
assert(plugboard.encode("A") == "A")

### Rotors
The number of possible combinations due to the plugboard is staggering, making brute force attempts to break a code extremely difficult. But using it alone would result in a simple substitution cipher – easily cracked with techniques like frequency analysis. The next step in the process, the rotors, allow the letter substitution to change mechanically every time a key is pressed, which prevents simple frequency analysis.

Over time, rotors with many different wiring patterns were developed, and different Enigma machines supported different types and numbers of rotors. The simulator follows this specification.

The Enigma machine supports *three or four* rotors and one reflector – notice that a reflector is really just a type of rotor where the characters line up in 13 perfect pairs. The rotors will be numbered *from right to left*, which is the “path” the current takes when first entering the rotors. The signal goes through each rotor in turn, hits the reflector, then goes through the rotors again in reverse order (left to right). When it goes through the rotors in reverse order, it uses the *reverse* wiring. So if A is mapped to L when going from right to left, then L is mapped to A on the reverse journey (of course it is not possible to actually hit the same wire on the way back – a letter cannot encode into itself in the machine as a whole).

Ignoring reflectors, which never rotate, there are two types of rotating rotor, based on whether or not the rotor contains a *notch*. If the rotor contains a notch, then when it is on a certain position and is rotated, it will cause the rotor in the next slot to rotate as well. In addition, in a four slot Engima machine, the leftmost (fourth) rotor never rotates. Rotation will be explained in more detail shortly.

Each rotor is chosen from a box containing seven possible wiring patterns. There are two rotors labelled `Beta` and `Gamma`. Then there are five rotors labelled with Roman numerals which do rotate: `I, II, III, IV, V`. The simulator also supports three possible reflector wiring patterns, labelled `A, B, C`.

Note: the wiring patterns are all real, taken from [this page](https://en.wikipedia.org/wiki/Enigma_rotor_details). The page contains more rotors; the ones listed here are supported. The following patterns all assume the rotors are in their default position, with their default ring setting, and going from right to left (the initial path).

<table><thead>
<tr><th></th><th colspan="26"><center>Mapping from letter</center></th></tr>
<tr><th style="text-align:left">Label</th><th>A</th><th>B</th><th>C</th><th>D</th><th>E</th><th>F</th><th>G</th><th>H</th><th>I</th><th>J</th><th>K</th><th>L</th><th>M</th><th>N</th><th>O</th><th>P</th><th>Q</th><th>R</th><th>S</th><th>T</th><th>U</th><th>V</th><th>W</th><th>X</th><th>Y</th><th>Z</th></tr></thead><tbody>
<tr><th style="text-align:left">Beta</th><td>L</td><td>E</td><td>Y</td><td>J</td><td>V</td><td>C</td><td>N</td><td>I</td><td>X</td><td>W</td><td>P</td><td>B</td><td>Q</td><td>M</td><td>D</td><td>R</td><td>T</td><td>A</td><td>K</td><td>Z</td><td>G</td><td>F</td><td>U</td><td>H</td><td>O</td><td>S</td></tr>
<tr><th style="text-align:left">Gamma</th><td>F</td><td>S</td><td>O</td><td>K</td><td>A</td><td>N</td><td>U</td><td>E</td><td>R</td><td>H</td><td>M</td><td>B</td><td>T</td><td>I</td><td>Y</td><td>C</td><td>W</td><td>L</td><td>Q</td><td>P</td><td>Z</td><td>X</td><td>V</td><td>G</td><td>J</td><td>D</td></tr>
<tr><th style="text-align:left">I</th><td>E</td><td>K</td><td>M</td><td>F</td><td>L</td><td>G</td><td>D</td><td>Q</td><td>V</td><td>Z</td><td>N</td><td>T</td><td>O</td><td>W</td><td>Y</td><td>H</td><td>X</td><td>U</td><td>S</td><td>P</td><td>A</td><td>I</td><td>B</td><td>R</td><td>C</td><td>J</td></tr>
<tr><th style="text-align:left">II</th><td>A</td><td>J</td><td>D</td><td>K</td><td>S</td><td>I</td><td>R</td><td>U</td><td>X</td><td>B</td><td>L</td><td>H</td><td>W</td><td>T</td><td>M</td><td>C</td><td>Q</td><td>G</td><td>Z</td><td>N</td><td>P</td><td>Y</td><td>F</td><td>V</td><td>O</td><td>E</td></tr>
<tr><th style="text-align:left">III</th><td>B</td><td>D</td><td>F</td><td>H</td><td>J</td><td>L</td><td>C</td><td>P</td><td>R</td><td>T</td><td>X</td><td>V</td><td>Z</td><td>N</td><td>Y</td><td>E</td><td>I</td><td>W</td><td>G</td><td>A</td><td>K</td><td>M</td><td>U</td><td>S</td><td>Q</td><td>O</td></tr>
<tr><th style="text-align:left">IV</th><td>E</td><td>S</td><td>O</td><td>V</td><td>P</td><td>Z</td><td>J</td><td>A</td><td>Y</td><td>Q</td><td>U</td><td>I</td><td>R</td><td>H</td><td>X</td><td>L</td><td>N</td><td>F</td><td>T</td><td>G</td><td>K</td><td>D</td><td>C</td><td>M</td><td>W</td><td>B</td></tr>
<tr><th style="text-align:left">V</th><td>V</td><td>Z</td><td>B</td><td>R</td><td>G</td><td>I</td><td>T</td><td>Y</td><td>U</td><td>P</td><td>S</td><td>D</td><td>N</td><td>H</td><td>L</td><td>X</td><td>A</td><td>W</td><td>M</td><td>J</td><td>Q</td><td>O</td><td>F</td><td>E</td><td>C</td><td>K</td></tr>
<tr><th style="text-align:left">A</th><td>E</td><td>J</td><td>M</td><td>Z</td><td>A</td><td>L</td><td>Y</td><td>X</td><td>V</td><td>B</td><td>W</td><td>F</td><td>C</td><td>R</td><td>Q</td><td>U</td><td>O</td><td>N</td><td>T</td><td>S</td><td>P</td><td>I</td><td>K</td><td>H</td><td>G</td><td>D</td></tr>
<tr><th style="text-align:left">B</th><td>Y</td><td>R</td><td>U</td><td>H</td><td>Q</td><td>S</td><td>L</td><td>D</td><td>P</td><td>X</td><td>N</td><td>G</td><td>O</td><td>K</td><td>M</td><td>I</td><td>E</td><td>B</td><td>F</td><td>Z</td><td>C</td><td>W</td><td>V</td><td>J</td><td>A</td><td>T</td></tr>
<tr><th style="text-align:left">C</th><td>F</td><td>V</td><td>P</td><td>J</td><td>I</td><td>A</td><td>O</td><td>Y</td><td>E</td><td>D</td><td>R</td><td>Z</td><td>X</td><td>W</td><td>G</td><td>C</td><td>T</td><td>K</td><td>U</td><td>Q</td><td>S</td><td>B</td><td>N</td><td>M</td><td>H</td><td>L</td></tr>
</tbody></table>

### Single Rotor
A first demonstration of the rotor concepts introduced so far. A rotor in its default position, with default ring setting, maps `A` to `E` going right to left (`I` wiring) and `A` to `U` going back left to right.


In [4]:
rotor = Rotor("I","A",1)
assert(rotor.encode_right_to_left("A") == "E")
assert(rotor.encode_left_to_right("A") == "U")

### The Enigma Machine
The next step is to model the remaining details of how the rotors work and to combine them into a single machine that performs the full encoding path.

#### Multiple Rotors
For now, let's ignore the plugboard, and introduce the remaining details for the rotors.

It is common to see the selection of rotors specified in a single sequence from left to right, as the operator would see when looking down, such as on a [German code book](https://en.wikipedia.org/wiki/Enigma_machine#/media/File:Enigma_keylist_3_rotor.jpg). For example, on the top row of that linked image, you can see the rotors should be `I V III`. This means the first (rightmost) rotor is `III`, and so on. 

The code book also specifies each rotor's “ring setting” – in that image you can see the rotors on the top line should be set to `14 09 24` correspondingly. 

The rotor settings are explained later; first, here is how the wiring works with multiple rotors. This is a worked example using three rotors, with all the settings in their default positions – this means you can read the character mappings directly from the table above.

Imagine a `III` rotor sat upright in the machine in the right hand position. The right hand side of the rotor has 26 *pins* and the left hand side has 26 *contacts*, one for each character. The pins on the right are connected to the contacts of the rotor housing which is wired into the plugboard. So if the plugboard sends a signal on the `A` contact of the housing, then this hits the `A` *pin* of the rotor, then this passes through the rotor's internal wiring (check the table for the `III` rotor) and we receive an output signal on the `B` *contact* of the rotor. This will now go into the next rotor: remember there are three or four rotors, followed by a reflector.

Suppose there is a `II` rotor in the middle position, a `I` rotor in the left position, and then a `B` reflector. Here is the full path when we send an `A` signal from the plugboard:
* `A` signal comes in
* `III` rotor receives signal on `A` pin, which connects to `B` contact
* `II` rotor receives signal on `B` pin, which connects to `J` contact
* `I` rotor receives signal on `J` pin, which connects to `Z` contact
* `B` reflector receives signal on `Z` connecting to `T` <br/>
  (now the signal goes backwards, hitting the *contacts* and coming out the *pins*)
* `I` rotor receives signal on `T` contact, which connects to `L` pin
* `II` rotor receives signal on `L` contact, which connects to `K` pin
* `III` rotor receives signal on `K` contact, which connects to `U` pin
* `U` signal is output

The final output for this `A` signal is a `U`. The path can be followed using the table above; things get more complicated with rotation.

#### Rotation
As mentioned, the rightmost rotor (first in the order of the electrical circuit) rotates at *the start* of each keypress, i.e. *before* the character signal is passed through the circuit. This is what makes the Enigma machine more powerful than a fixed substitution cipher, one rotation causes a completely different circuit and substitution.

The rotation of the rotor advances a setting called the ***position*** of each rotor, which is visible through a window on the machine. This setting is marked on the rotor and is a character between `A` and `Z`, but it is helpful to think of this number as an *offset* rather than a *letter*. In the example above we assumed all of the rotors were set to position `A`. Rotating moves the *pins and the contacts* up by one position (`A` becomes `B`, etc).

We mentioned before that if we input `A` into the `III` rotor in its default position (which is labelled `A`) then it produces an output of `B`. In our full example we assumed that the rotors were set to `AAA`, but if we set them this way on the machine, then as soon as we press the `A` key the rightmost rotor would rotate giving `AAB`, and this is the circuit that would be made (rotation always happens first). Here is what happens in this setting.

Now if we input an `A` signal from the plugboard, it will come out of the `A` contact of the housing, but it will hit the `B` pin of the rotor due to its rotation by one position. The `B` *pin* is wired to the `D` *contact* (reading from the table).

***But*** the `D` contact has also been rotated one position inside the machine, so it actually lines up with the `C` *pin* of the next rotor in its default position. The other two rotors and the reflector work as normal, and on the way back rotor `II` sends a signal on its `E` pin. Since `III` is rotated, this hits the `F` contact which is wired to the `C` pin. But again since it is rotated, this his the `B` contact of the rotor housing, and is what is sent back to the plugboard.

Here is the full example again, now assuming the rotors are set to `AAB` instead of `AAA`:
* `A` signal comes in
* `III (B)` rotor receives signal on `B` pin, which connects to `D` contact 
* `II (A)` rotor receives signal on `C` pin, which connects to `D` contact
* `I (A)` rotor receives signal on `D` pin, which connects to `F` contact
* `B` reflector receives signal on `F` connecting to `S`
* `I (A)` rotor receives signal on `S` contact, which connects to `S` pin
* `II (A)` rotor receives signal on `S` contact, which connects to `E` pin
* `III (B)` rotor receives signal on `F` contact, which connects to `C` pin
* `B` signal is output

You can try using the Enigma machine emulator [on this page](https://www.101computing.net/enigma-machine-emulator/). It defaults to the same settings: rotors `I II III` all initially set to position `A`, so when you press the `A` key on the keyboard you should get `B`.

The path can be followed using the table of wirings above.

#### Ring Settings
In addition each rotor could be configured by changing the *ring setting*, which is a fixed offset that would apply between the internal wiring and the external markings. The ring settings were either given from `A-Z` or `01-26` – you saw the latter in the code book image linked above, and we'll use these too to avoid confusion with the *position* setting.

If a rotor's ring setting is set to `01` then nothing is changed, the wiring is exactly as written in the table above.

*Increasing* the ring setting has the exact same effect as *decreasing* the position setting. It shifts the internal wiring in the opposite direction.

Earlier we saw that `A` becomes `U` with the given rotors actually set to `AAA` – we'd have to start on `AAZ` to get this result on the machine, since the rotation happens first (try it on the emulator). Alternatively, we could set the ring position to `02` on the rightmost rotor and set the initial positions to `AAA`, and we'll get the same result for a single press (again you can try this on the emulator; it uses letters for ring settings, if you click the rotor you can set the ring setting to `B`).

If we keep pressing `A`, the two configurations will produce many of the same characters, but not always. We have detailed how the rightmost rotor rotates, but not the others, and this detail will eventually produce a difference between the two sets of settings above.

#### Turnover
The rotors labelled `I` to `V` have *notches*. If a rotor has a notch and is currently set to its notch position, then it will turn the next rotor on the next keypress (this is called *turnover*). Here are the notch positions:

<table><thead><tr><th>Rotor</th><th>Notch</th></tr></thead><tbody><tr><td style="text-align:center">I</td><td style="text-align:center">Q</td></tr><tr><td style="text-align:center">II</td><td style="text-align:center">E</td></tr><tr><td style="text-align:center">III</td><td style="text-align:center">V</td></tr><tr><td style="text-align:center">IV</td><td style="text-align:center">J</td></tr><tr><td style="text-align:center">V</td><td style="text-align:center">Z</td></tr></tbody></table>

So if a `II` rotor in the first, rightmost, slot of the machine is *currently set to* position `E` then when you press a key the first rotor will turn to position `F` *and* the rotor in the second position will turn as well, and then the electrical signal will be sent through the circuit.

If the `II` rotor had been in the second position, then it will obviously turn much more slowly. But if its position is set to `E` and a key is pressed then it will rotate and turn the rotor in the third position also. 

There is an important detail here called the *double step*. Normally the second rotor will only turn once every 26 turns of the rightmost rotor. But if the second rotor is *on* its notch setting, then it will turn again *as it turns the third rotor*. Obviously the second rotor must have just rotated to land on its notch, so it actually rotates for two keypresses in a row.


Suppose we have the rotors `I II III`, on positions `A C U`. Pressing a key turns the `III` rotor and we get `A C V`. Now the `III` rotor is on its notch, so pressing a key also turns `II` and we get `A D W`. If we continue pressing keys we'll get `A D X`, `A D Y`, `A D Z`, `A D A`, and so on. Several keypresses later we wrap round again and approach the notch on `III` again on setting `A D U`. When we press once we get `A D V`. Now `III` is on its notch so pressing again turns `II` and we get `A E W`. But now `II` is on its notch, so when we press again *all three* rotors rotate and we get `B F X` – `III` turns because it always turns, `II` turns because it is on its notch, and `I` turns because `II` turned on its notch (turnover).

Notice the *ring setting* is inconsequential in this turnover process – it only requires the current position to line up with the notch setting.

The four-rotor machines did not have a additional lever, and so whether the third rotor had a notch or not the fourth rotor would not turn. In addition if a notchless rotor (e.g. `Beta`) was in the first position, then it will never cause the second rotor to rotate. The rightmost rotor will still always rotate exactly once on every keypress. Notchless rotors can still be set to different position settings as part of the setup process.

If you are curious about the mechanism, you can see a video of a mock-up version of an Enigma machine in action [here](https://www.youtube.com/watch?v=hcVhQeZ5gI4), showing how the the ratchets, levers, and notches contribute towards the rotation on each keypress, and also demonstrating the double step (around 26 seconds into the video).

### Multiple Rotors
The cell below checks that the rotors work together, covering these cases:
* With rotors `I II III`, reflector `B`, ring settings `01 01 01`, and initial positions `A A Z`, encoding an `A` produces a `U`.
* With rotors `I II III`, reflector `B`, ring settings `01 01 01`, and initial positions `A A A`, encoding an `A` produces a `B`.
* With rotors `I II III`, reflector `B`, ring settings `01 01 01`, and initial positions `Q E V`, encoding an `A` produces an `L`.
* With rotors `IV V Beta`, reflector `B`, ring settings `14 09 24`, and initial positions `A A A`, encoding an `H` produces a `Y`.
* With rotors `I II III IV`, reflector `C`, ring settings `07 11 15 19`, and initial positions `Q E V Z`, encoding a `Z` produces a `V`.


In [5]:
enigma = EnigmaMachine(rotors=[Rotor("I","A",1), Rotor("II","A",1), Rotor("III","Z",1)], reflector=Rotor("B","A",1))
enigma._init_rotors()
assert(enigma.encode_character("A") == "U")

enigma = EnigmaMachine(rotors=[Rotor("I","A",1), Rotor("II","A",1), Rotor("III","A",1)], reflector=Rotor("B","A",1))
enigma._init_rotors()
assert(enigma.encode_character("A") == "B")

enigma = EnigmaMachine(rotors=[Rotor("I","Q",1), Rotor("II","E",1), Rotor("III","V",1)], reflector=Rotor("B","A",1))
enigma._init_rotors()
assert(enigma.encode_character("A") == "L")

enigma = EnigmaMachine(
            rotors=[Rotor("IV","A",14), Rotor("V","A",9), Rotor("Beta","A",24)],
            reflector=Rotor("B","A",1)
        )
enigma._init_rotors()
assert(enigma.encode_character("H") == "Y")

enigma = EnigmaMachine(rotors=[Rotor("I","Q",7), Rotor("II","E",11), Rotor("III","V",15), Rotor("IV","Z",19)], reflector=Rotor("C","A",1))
enigma._init_rotors()
assert(enigma.encode_character("Z") == "V")

### Enigma Machine Demonstration
### Full Enigma Machine
Putting all of the elements together: the machine supports a plugboard with up to 10 leads, three or four rotors, and a reflector. It encodes an entire string of characters made from the letters `A`-`Z`, correctly advancing the rotors.

The cell below runs two examples.
#### Example 1
Set up your enigma machine with rotors `I II III`, reflector `B`, ring settings `01 01 01`, and initial positions `A A Z`.

The plugboard should map the following pairs: `HL MO AJ CX BZ SR NI YW DG PK`.

*The result of encoding the string* `HELLOWORLD` *should be* `RFKTMBXVVW`.

#### Example 2
Set up your enigma machine with rotors `IV V Beta I`, reflector `A`, ring settings `18 24 03 05`, and initial positions `E Z G P`. 

The plugboard should map the following pairs: `PC XZ FM QA ST NB HY OR EV IU`.

*Find the result of decoding the following string:* `BUPXWJCDPFASXBDHLBBIBSRNWCSZXQOLBNXYAXVHOGCUUIBCVMPUZYUUKHI`.


In [6]:
# EXAMPLE 1

# PlugBoard Init

plugboard = Plugboard()
plugboard.add(PlugLead("HL"))
plugboard.add(PlugLead("MO"))
plugboard.add(PlugLead("AJ"))
plugboard.add(PlugLead("CX"))
plugboard.add(PlugLead("BZ"))
plugboard.add(PlugLead("SR"))
plugboard.add(PlugLead("NI"))
plugboard.add(PlugLead("YW"))
plugboard.add(PlugLead("DG"))
plugboard.add(PlugLead("PK"))

# Enigma Machine Init

enigma = EnigmaMachine(rotors=[Rotor("I","A",1), Rotor("II","A",1), Rotor("III","Z",1)], reflector=Rotor("B","A",1), plugboard = plugboard)
enigma._init_rotors()

new_word = ""
for w in "HELLOWORLD":
    new_w = enigma.encode_character(w)
    new_word += new_w

print(f"The word 'HELLOWORLD' becomes: {new_word}")

# EXAMPLE 2

#PlugBoard Init
plugboard = Plugboard()
plugboard.add(PlugLead("PC"))
plugboard.add(PlugLead("XZ"))
plugboard.add(PlugLead("FM"))
plugboard.add(PlugLead("QA"))
plugboard.add(PlugLead("ST"))
plugboard.add(PlugLead("NB"))
plugboard.add(PlugLead("HY"))
plugboard.add(PlugLead("OR"))
plugboard.add(PlugLead("EV"))
plugboard.add(PlugLead("IU"))

#Enigma Machine Init

enigma = EnigmaMachine(rotors=[Rotor("IV","E",18), Rotor("V","Z",24), Rotor("Beta","G",3), Rotor("I","P",5)], reflector=Rotor("A","A",1), plugboard = plugboard)
enigma._init_rotors()

new_word = ""
for w in "BUPXWJCDPFASXBDHLBBIBSRNWCSZXQOLBNXYAXVHOGCUUIBCVMPUZYUUKHI":
    new_w = enigma.encode_character(w)
    new_word += new_w

print(f"The word 'BUPXWJCDPFASXBDHLBBIBSRNWCSZXQOLBNXYAXVHOGCUUIBCVMPUZYUUKHI' is: {new_word}")

The word 'HELLOWORLD' becomes: RFKTMBXVVW
The word 'BUPXWJCDPFASXBDHLBBIBSRNWCSZXQOLBNXYAXVHOGCUUIBCVMPUZYUUKHI' is: CONGRATULATIONSONPRODUCINGYOURWORKINGENIGMAMACHINESIMULATOR


## Part Two – Code Breaking
In this part the simulator is used to break messages encrypted with an Enigma machine. For each message we have the ciphertext, a *crib* (a word we expect to appear in the original text), and partial information about the machine. The goal is to recover the original plaintext and the full machine settings.

The Bletchley codebreakers combined weaknesses in the machine's encryption, mathematical techniques, and computing power to solve German codes. Here only computing power is used: the missing settings are *brute-forced*, trying each candidate until the output contains the crib.

The number of possible settings for Enigma is far too vast for brute force alone, but the partial information narrows the search to something feasible on a modern computer: each message takes at most a few minutes.

The code that cracks them is in `code_breaking.py`. Running `python code_breaking.py` reproduces the results listed below without manual intervention.

### Codes
Each code contains the ciphertext (the encrypted text), and a crib: a word or phrase that you think appears exactly *somewhere* within the original text.

You should use the usual rotors and reflectors specified in the table above unless the question specifies otherwise. So if the question does not specify a rotor, the valid options are only `Beta`, `Gamma`, `I`, `II`, `III`, `IV`, and `V`, the valid options for reflectors are just `A`, `B`, and `C`.

The machines in this section will only ever use 3 rotors – meaning 3 ring settings and 3 starting positions also.

More than one set of Enigma machine settings can produce an output containing the crib word. In that case the correct one is chosen by reading the output.

#### Code 1
The machine is set up in that day's position, but the reflector label is missing. All the other settings are known. The message might contain the word "SECRETS".

* Code: `DMEXBMKYCVPNQBEDHXVPZGKMTFFBJRPJTLHLCHOTKOYXGGHZ`
* Crib: `SECRETS`


* Rotors: `Beta Gamma V`
* Reflector: Unknown
* Ring settings: `04 02 14`
* Starting positions: `MJM`
* Plugboard pairs: `KI XN FL`

#### Code 2
The day's settings are known, but the initial rotor positions change for each message, so the starting positions are unknown. The message is addressed to a university.

* Code: `CMFSUPKNCBMUYEQVVDYKLRQZTPUFHSWWAKTUGXMPAMYAFITXIJKMH`
* Crib: `UNIVERSITY`


* Rotors: `Beta I III`
* Reflector: `B`
* Ring settings: `23 02 10`
* Starting positions: Unknown
* Plugboard pairs: `VH PT ZG BJ EY FS`

#### Code 3
The sender is known to only use even numbers: no odd-numbered rotor (ruling out `I`, `III`, and `V`) and no *ring setting* containing an odd digit (`02` is allowed, `11` and `12` are not). The message is expected to contain the word "THOUSANDS".

* Code: `ABSKJAKKMRITTNYURBJFWQGRSGNNYJSDRYLAPQWIAGKJYEPCTAGDCTHLCDRZRFZHKNRSDLNPFPEBVESHPY`
* Crib: `THOUSANDS`


* Rotors: Unknown but restricted (see above)
* Reflector: Unknown
* Ring settings: Unknown but restricted (see above)
* Starting positions: `EMY`
* Plugboard pairs: `FH TS BE UQ KD AL`

#### Code 4
Two plug leads are partially missing (marked with question marks in the settings below), and the message is expected to mention a tutor.

* Code: `SDNTVTPHRBNWTLMZTQKZGADDQYPFNHBPNHCQGBGMZPZLUAVGDQVYRBFYYEIXQWVTHXGNW`
* Crib: `TUTOR`


* Rotors: `V III IV`
* Reflector: `A`
* Ring settings: `24 12 10`
* Starting positions: `SWU`
* Plugboard pairs: `WP RJ A? VF I? HN CG BS`

#### Code 5
The message is probably spam related to a social media platform. Most settings are known, but it was made with a non-standard reflector: a standard reflector in which two pairs of wires have been modified.

To be clear, a single wire connects two letters, e.g. mapping `A` to `Y` and `Y` to `A`. The sender has taken two wires (four letters in two pairs), e.g. `A-Y` and `H-J`, and swapped one of the ends, so one option would be `H-Y` and `A-J`. They did this twice, so they modified eight letters total (they did not swap the same wire more than once). 

The result must report the original reflector and the modifications.

* Code: `HWREISXLGTTBYVXRCWWJAKZDTVZWKBDJPVQYNEQIOTIFX`
* Crib: the name of a social media website/platform


* Rotors: `V II IV`
* Reflector: Unknown and non-standard (see above)
* Ring settings: `06 18 07`
* Starting positions: `AJL`
* Plugboard pairs: `UG IE PO NX WT`

### Results

#### Code 1:

Word: NICEWORKYOUVEMANAGEDTODECODETHEFIRSTSECRETSTRING found with Reflector C.

#### Code 2:

Word: IHOPEYOUAREENJOYINGTHEUNIVERSITYOFBATHEXPERIENCESOFAR found with Starting Positions: ['I', 'M', 'G'].

#### Code 3:

Word: SQUIRRELSPLANTTHOUSANDSOFNEWTREESEACHYEARBYMERELYFORGETTINGWHERETHEYPUTTHEIRACORNS found with Rotors: ['II', 'Gamma', 'IV'], Reflector: C, and RingSettings: [24, 8, 20].

#### Code 4:

Word NOTUTORSWEREHARMEDNORIMPLICATEDOFCRIMESDURINGTHEMAKINGOFTHESEEXAMPLES found with first PlugLead: AT, and second PlugLeaf: IK.

#### Code 5:

Word: YOUCANFOLLOWMYDOGONINSTAGRAMATTALESOFHOFFMANN found with Reflector: B with the following modifications:

- (('A', 'P'), ('Y', 'I'));
- (('B', 'Q'), ('R', 'E')).

The resultant New Reflector is: 

{'C': 'U', 'U': 'C', 'D': 'H', 'H': 'D', 'F': 'S', 'S': 'F', 'G': 'L', 'L': 'G', 'J': 'X', 'X': 'J', 'K': 'N', 'N': 'K', 'M': 'O', 'O': 'M', 'T': 'Z', 'Z': 'T', 'V': 'W', 'W': 'V', 'A': 'P', 'P': 'A', 'Y': 'I', 'I': 'Y', 'B': 'Q', 'Q': 'B', 'R': 'E', 'E': 'R'}

## Extras


### Logging

The module uses a lightweight logging setup based on Python's standard `logging` library. Messages are organised across three severity levels so that output stays minimal by default: `WARNING` reports unexpected conditions such as a full plugboard, rejected or duplicate plug leads, and invalid letter mappings; `INFO` records one-off setup events like machine initialisation and the chosen configuration; and `DEBUG` provides detailed, per-character encoding traces intended for in-depth troubleshooting. The verbosity is governed by a single `basicConfig` level, allowing the amount of logging to be adjusted without modifying any code — the detailed traces remain hidden unless the level is explicitly lowered to `DEBUG`.